In [4]:
# Step 1: Imports

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [5]:
# Step 2: Sample dataset
# This data follows the same structure as the
# MovieLens dataset format (userId, movieId, title, genres, rating).

data = {
    "userId":  [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5],
    "movieId": [1, 2, 3, 1, 4, 5, 2, 3, 6, 1, 5, 6, 4, 6, 2],
    "title": [
        "Toy Story", "Jumanji", "Grumpier Old Men",
        "Toy Story", "Waiting to Exhale", "Father of the Bride Part II",
        "Jumanji", "Grumpier Old Men", "Heat",
        "Toy Story", "Father of the Bride Part II", "Heat",
        "Waiting to Exhale", "Heat", "Jumanji"
    ],
    "genres": [
        "Animation|Comedy|Family", "Adventure|Fantasy", "Comedy|Romance",
        "Animation|Comedy|Family", "Comedy|Drama", "Comedy",
        "Adventure|Fantasy", "Comedy|Romance", "Action|Crime|Thriller",
        "Animation|Comedy|Family", "Comedy", "Action|Crime|Thriller",
        "Comedy|Drama", "Action|Crime|Thriller", "Adventure|Fantasy"
    ],
    "rating": [4.0, 3.5, 2.0, 5.0, 3.0, 3.5, 4.5, 2.5, 4.0, 3.0, 4.0, 4.5, 2.5, 5.0, 3.0]
}

df = pd.DataFrame(data)
df.head()


,userId,movieId,title,genres,rating
0,1,1,Toy Story,Animation|Comedy|Family,4.0
1,1,2,Jumanji,Adventure|Fantasy,3.5
2,1,3,Grumpier Old Men,Comedy|Romance,2.0
3,2,1,Toy Story,Animation|Comedy|Family,5.0
4,2,4,Waiting to Exhale,Comedy|Drama,3.0


In [3]:
# Step 3: Content based filtering
# This section builds a TF-IDF matrix from movie genres and calculates
# cosine similarity between movies. This approach is based on the common
# tutorial pattern used for beginner movie recommendation projects
# (Banik, 2018; Kaggle, n.d.), credited fully in the report references.

movies = df[["movieId", "title", "genres"]].drop_duplicates(subset="movieId").reset_index(drop=True)

tfidf = TfidfVectorizer(token_pattern=r"[A-Za-z]+")
genre_matrix = tfidf.fit_transform(movies["genres"].str.replace("|", " "))

content_sim = cosine_similarity(genre_matrix)
movie_index = pd.Series(movies.index, index=movies["movieId"])

def content_based_recommend(movie_id, top_n=5):
    """Return the most similar movies to a given movie, based on genre similarity."""
    if movie_id not in movie_index:
        return pd.DataFrame()
    idx = movie_index[movie_id]
    sims = list(enumerate(content_sim[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)
    sims = [s for s in sims if s[0] != idx][:top_n]
    result_idx = [s[0] for s in sims]
    scores = [s[1] for s in sims]
    out = movies.iloc[result_idx].copy()
    out["content_score"] = scores
    return out

# Quick test
content_based_recommend(1)


,movieId,title,genres,content_score
4,5,Father of the Bride Part II,Comedy,0.386839
2,3,Grumpier Old Men,Comedy|Romance,0.197375
3,4,Waiting to Exhale,Comedy|Drama,0.197375
1,2,Jumanji,Adventure|Fantasy,0.000000
5,6,Heat,Action|Crime|Thriller,0.000000


In [6]:
# Step 4: Collaborative filtering
# Full hybrid system builds a user-item matrix
# and calculates similarity between users using a centered cosine method,
# similar to Pearson correlation (Sarwar et al., 2001).

user_item = df.pivot_table(index="userId", columns="movieId", values="rating")

def user_similarity(user_id):
    """Calculate similarity between the given user and all other users."""
    matrix = user_item.fillna(0)
    if user_id not in matrix.index:
        return pd.Series(dtype=float)
    centered = matrix.sub(matrix.mean(axis=1), axis=0)
    sims = cosine_similarity(centered)
    sims_df = pd.DataFrame(sims, index=matrix.index, columns=matrix.index)
    return sims_df[user_id].drop(user_id)

def collaborative_recommend(user_id, top_n=5):
    """Predict ratings for movies a user has not seen, using similar users."""
    if user_id not in user_item.index:
        return pd.DataFrame()

    sims = user_similarity(user_id)
    if sims.empty or sims.abs().sum() == 0:
        return pd.DataFrame()

    user_ratings = user_item.loc[user_id]
    unseen = user_ratings[user_ratings.isna()].index

    predictions = {}
    for movie_id in unseen:
        others = user_item[movie_id].dropna()
        weights = sims.loc[others.index]
        if weights.abs().sum() == 0:
            continue
        pred = np.dot(others, weights) / weights.abs().sum()
        predictions[movie_id] = pred

    if not predictions:
        return pd.DataFrame()

    pred_series = pd.Series(predictions).sort_values(ascending=False).head(top_n)
    out = movies.set_index("movieId").loc[pred_series.index].reset_index()
    out["collab_score"] = pred_series.values
    return out

# Quick test
collaborative_recommend(1)


,index,title,genres,collab_score
0,4,Waiting to Exhale,Comedy|Drama,-1.303388
1,5,Father of the Bride Part II,Comedy,-2.347647
2,6,Heat,Action|Crime|Thriller,-2.883400


In [8]:
# Step 5: Hybrid recommendation function
# This blends content based and collaborative filtering scores together
# using an adjustable weight (alpha), following the hybrid approach
# described by Burke (2002). It also handles the cold start problem for
# new users by falling back to popularity based recommendations.

def hybrid_recommend(user_id, seed_movie_id=None, top_n=5, alpha=0.6):
    collab = collaborative_recommend(user_id, top_n=top_n * 2)

    if collab.empty:
        # Cold start fallback: recommend based on overall popularity
        popularity = df.groupby("movieId")["rating"].mean().sort_values(ascending=False)
        top_movies = movies.set_index("movieId").loc[popularity.index[:top_n]].reset_index()
        top_movies["hybrid_score"] = popularity.values[:top_n]
        return top_movies

    # Fix: rename the 'index' column back to 'movieId' if it exists in collab
    if "index" in collab.columns and "movieId" not in collab.columns:
        collab = collab.rename(columns={"index": "movieId"})

    if seed_movie_id is None:
        rated = user_item.loc[user_id].dropna()
        seed_movie_id = rated.idxmax() if not rated.empty else movies.iloc[0]["movieId"]

    content = content_based_recommend(seed_movie_id, top_n=top_n * 2)

    merged = pd.merge(collab, content, on=["movieId", "title", "genres"], how="outer")
    merged["collab_score"] = merged["collab_score"].fillna(0)
    merged["content_score"] = merged["content_score"].fillna(0)
    merged["hybrid_score"] = alpha * merged["collab_score"] + (1 - alpha) * merged["content_score"]

    merged = merged.sort_values("hybrid_score", ascending=False).head(top_n)
    return merged[["movieId", "title", "genres", "hybrid_score"]]

# Quick test
hybrid_recommend(1)

,movieId,title,genres,hybrid_score
1,3,Grumpier Old Men,Comedy|Romance,0.078950
0,2,Jumanji,Adventure|Fantasy,0.000000
2,4,Waiting to Exhale,Comedy|Drama,-0.703083
3,5,Father of the Bride Part II,Comedy,-1.253853
4,6,Heat,Action|Crime|Thriller,-1.730040


In [9]:
# Step 6: Add a new rating and rebuild models
# This allows a user to rate a movie during the session, after which the
# data and models are rebuilt so the new rating is reflected immediately.

def add_rating(user_id, movie_id, title, genres, rating):
    global df, movies, user_item, content_sim, movie_index, genre_matrix

    new_row = {"userId": user_id, "movieId": movie_id, "title": title, "genres": genres, "rating": rating}
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    movies = df[["movieId", "title", "genres"]].drop_duplicates(subset="movieId").reset_index(drop=True)
    genre_matrix = tfidf.fit_transform(movies["genres"].str.replace("|", " "))
    content_sim = cosine_similarity(genre_matrix)
    movie_index = pd.Series(movies.index, index=movies["movieId"])

    user_item = df.pivot_table(index="userId", columns="movieId", values="rating")
    print(f"Rating recorded: user {user_id} rated '{title}' as {rating}")


In [13]:
# Step 7: Conversational interface
# This runs a simple text based chatbot loop so the user interacts with
# the system by typing commands instead of using a graphical interface.

def print_movies(result_df):
    if result_df.empty:
        print("No recommendations available.")
        return
    for _, row in result_df.iterrows():
        score_col = [c for c in result_df.columns if "score" in c]
        score = f" (score: {row[score_col[0]]:.2f})" if score_col else ""
        print(f"  - {row['title']} [{row['genres']}]{score}")

print("=== Movie Recommendation Assistant ===")
print("Commands: recommend <userId> | rate <userId> <movieId> <rating> | list | exit")

while True:
    print("=" * 60)
    command = input("User Input : ").strip().lower()

    if command == "exit":
        print("Assistant: Goodbye!")
        break

    elif command == "list":
        print(movies[["movieId", "title", "genres"]].to_string(index=False))

    elif command.startswith("recommend"):
        parts = command.split()
        try:
            user_id = int(parts[1])
        except (IndexError, ValueError):
            print("Assistant: Please use the format: recommend <userId>")
            continue
        print(f"Assistant: Here are recommendations for user {user_id}:")
        results = hybrid_recommend(user_id)
        print_movies(results)

    elif command.startswith("rate"):
        parts = command.split()
        try:
            user_id, movie_id, rating = int(parts[1]), int(parts[2]), float(parts[3])
        except (IndexError, ValueError):
            print("Assistant: Please use the format: rate <userId> <movieId> <rating>")
            continue
        match = movies[movies["movieId"] == movie_id]
        if match.empty:
            print("Assistant: Unknown movieId. Type 'list' to see available movies.")
            continue
        title, genres = match.iloc[0]["title"], match.iloc[0]["genres"]
        add_rating(user_id, movie_id, title, genres, rating)

    else:
        print("Opps! Sorry, I did not understand that. Try: recommend, rate, list, or exit.")


=== Movie Recommendation Assistant ===
Commands: recommend <userId> | rate <userId> <movieId> <rating> | list | exit
User Input : list
 movieId                       title                  genres
       1                   Toy Story Animation|Comedy|Family
       2                     Jumanji       Adventure|Fantasy
       3            Grumpier Old Men          Comedy|Romance
       4           Waiting to Exhale            Comedy|Drama
       5 Father of the Bride Part II                  Comedy
       6                        Heat   Action|Crime|Thriller
User Input : recommend 1
Assistant: Here are recommendations for user 1:
  - Waiting to Exhale [Comedy|Drama] (score: 0.07)
  - Toy Story [Animation|Comedy|Family] (score: 0.00)
  - Jumanji [Adventure|Fantasy] (score: 0.00)
  - Grumpier Old Men [Comedy|Romance] (score: 0.00)
  - Father of the Bride Part II [Comedy] (score: -0.09)
User Input : rate 1 6 4.5
Rating recorded: user 1 rated 'Heat' as 4.5
User Input : recommend 9
Assistant: 